# Adverderial attacks for class imbalance lending default datasets

In credit default predictions, financial institution work with highly class imbalanced datsets which require resampling in order to improve prediction of the positive class (default). The nature of the business case make the model susceptible to adverserial attacks because lenders with a high risk (predicted to default) will want to modify the features they have control in order to flip the prediction to either receive the lending or lower interest rate on the lending. A defence is to train the model with adversarial hardened examples but this is at the cost of accuracy, with this notebook we want to see if certain resampling models can adversarially 'harden' or 'weaken'models.

Authors:
- Tim Angevare
- Rick Hezeman 
- Ruben Stevenaar
- Roy Mulder

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42 #for reproducibility


%matplotlib inline

Temporary Telegram updates

In [ ]:
import os
import requests

BOT_TOKEN = os.environ["TELEGRAM_BOT_TOKEN"]
CHAT_ID = os.environ["TELEGRAM_CHAT_ID"]

TG_API = "https://api.telegram.org/bot{token}/{method}"


def tg_api(token: str, method: str, **params) -> dict:
    url = TG_API.format(token=token, method=method)
    resp = requests.post(url, json=params, timeout=30)
    data = resp.json()
    if not data.get("ok"):
        raise RuntimeError(f"Telegram API error: {data.get('description', data)}")
    return data

def send_telegram_message(message: str):
    tg_api(BOT_TOKEN, "sendMessage", chat_id=CHAT_ID, text=message, parse_mode="Markdown")

In [ ]:
send_telegram_message("Adversarial attacks notebook started.")

Loading the accepted lending dataset.

"This dataset contains the full LendingClub data available from their site. There are separate files for accepted and rejected loans. The accepted loans also include the FICO scores, which can only be downloaded when you are signed in to LendingClub and download the data." https://www.kaggle.com/datasets/wordsforthewise/lending-club

In [ ]:
df = pd.read_csv("data/accepted.csv", low_memory=False)

## Section 1 exploring the dataset

In [ ]:
print(f"Number of Records: {df.shape[0]:,.0f}")
print(f"Number of Features: {df.shape[1]}")

In [ ]:
df.loan_status.value_counts()

For this research we only care about paid off and defaulted a.k.a charged off

In [ ]:
df = df[df.loan_status.isin(['Fully Paid', 'Charged Off'])]

In [ ]:
print("Columns:")
for col in df.columns:
    print(f" - {col}")

In [ ]:
df.head()

In [ ]:
missing = df.isnull().sum()
missing

We see a lot of nan values so lets drop the features that have a lot of NaN values

In [ ]:
missing_percent = (missing / len(df)) * 100

missing_table = pd.DataFrame({
    'Missing Values': missing,
    'Percentage (%)': missing_percent
})

missing_table = missing_table.sort_values(by='Percentage (%)', ascending=False)
pd.set_option('display.max_rows', None)

missing_table

In [ ]:
cols_to_drop = missing_table[missing_table['Percentage (%)'] > 50].index # Selects columsn with > 50% missing and returns their names

print("Columns to drop:")
for col in cols_to_drop:
    print(f" - {col}")

df = df.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
df.shape

Following the Kaggle notebooks we will keep the following collumns

| # | Feature | Description |
|---|---------|-------------|
| 0 | `loan_amnt` | The listed amount of the loan applied for by the borrower. If at some point in time, the credit department reduces the loan amount, then it will be reflected in this value. |
| 1 | `term` | The number of payments on the loan. Values are in months and can be either 36 or 60. |
| 2 | `int_rate` | Interest Rate on the loan |
| 3 | `installment` | The monthly payment owed by the borrower if the loan originates. |
| 4 | `grade` | LC assigned loan grade |
| 5 | `sub_grade` | LC assigned loan subgrade |
| 6 | `emp_length` | Employment length in years. Possible values are between 0 and 10 where 0 means less than one year and 10 means ten or more years. |
| 7 | `home_ownership` | The home ownership status provided by the borrower during registration or obtained from the credit report. Values: RENT, OWN, MORTGAGE, OTHER |
| 8 | `annual_inc` | The self-reported annual income provided by the borrower during registration. |
| 9 | `verification_status` | Indicates if income was verified by LC, not verified, or if the income source was verified |
| 10 | `issue_d` | The month which the loan was funded |
| 11 | `loan_status` | Current status of the loan |
| 12 | `purpose` | A category provided by the borrower for the loan request. |
| 13 | `title` | The loan title provided by the borrower |
| 14 | `zip_code` | The first 3 numbers of the zip code provided by the borrower in the loan application. |
| 15 | `addr_state` | The state provided by the borrower in the loan application |
| 16 | `dti` | A ratio calculated using the borrower's total monthly debt payments on the total debt obligations, excluding mortgage and the requested LC loan, divided by the borrower's self-reported monthly income. |
| 17 | `earliest_cr_line` | The month the borrower's earliest reported credit line was opened |
| 18 | `open_acc` | The number of open credit lines in the borrower's credit file. |
| 19 | `pub_rec` | Number of derogatory public records |
| 20 | `revol_bal` | Total credit revolving balance |
| 21 | `revol_util` | Revolving line utilization rate, or the amount of credit the borrower is using relative to all available revolving credit. |
| 22 | `total_acc` | The total number of credit lines currently in the borrower's credit file |
| 23 | `initial_list_status` | The initial listing status of the loan. Possible values are W, F |
| 24 | `application_type` | Indicates whether the loan is an individual application or a joint application with two co-borrowers |
| 25 | `mort_acc` | Number of mortgage accounts. |
| 26 | `pub_rec_bankruptcies` | Number of public record bankruptcies. |

In [ ]:
df = df[[
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length',
    'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
    'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
    'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc',
    'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status',
    'application_type', 'mort_acc', 'pub_rec_bankruptcies'
]]

In [ ]:
df.describe()

In [ ]:
df.dtypes

We ant to predict the rate of default so we wil rename it to target

In [ ]:
df = df.rename(columns={"loan_status": "target"})
df["target"] = df["target"].map({"Charged Off": 1, "Fully Paid": 0})


# Visualisation of the dataset

In [ ]:
df.groupby("target").count()

Lets look at the Default rate by Loan Purpose

In [ ]:
purpose_default_rate = (
    df.groupby("purpose")["target"].mean().sort_values(ascending=False) * 100
)

plt.figure(figsize=(10, 5))
purpose_default_rate.plot(kind="bar")

plt.title("Default Rate by Loan Purpose")
plt.xlabel("Loan purpose")
plt.ylabel("Default rate (%)")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

What is the default rate per grade?

In [ ]:
grade_default_rate = (
    df
    .groupby("grade")["target"]
    .mean()
    .sort_index() * 100
)

plt.figure(figsize=(6, 4))
grade_default_rate.plot(kind="bar")

plt.title("Default Rate by LendingClub Grade")
plt.xlabel("Grade")
plt.ylabel("Default rate (%)")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='viridis')
plt.show()

In [ ]:
boxplot = sns.boxplot(x="target", y="loan_amnt", data=df)
boxplot.set_title("Loan Amount Distribution by Target")
plt.show()

In [ ]:
installment = sns.histplot(data=df, x="installment", hue="target", kde=True, bins=30)
installment.set_title("Installment Distribution by Target")

plt.show()

interest_rate = sns.histplot(data=df, x="int_rate", hue="target", kde=True, bins=30)
interest_rate.set_title("Interest Rate Distribution by Target")

plt.show()

In [ ]:
df.corr(numeric_only=True)['target'].drop('target').sort_values().plot(kind='bar',
    title="Correlation between Loan status and Numeric Features", 
    ylabel='Correlation', xlabel='Numerical Features', 
)
plt.show()

# Section 2 DPT

### Convert categorical features into numereric

In [ ]:
df.dtypes

In [ ]:
df.application_type.unique()

Since `application_type` is just two values we binary encode

In [ ]:
df['Individual application'] = df['application_type'] == 'INDIVIDUAL'
df.drop(columns=['application_type'], inplace=True)

In [ ]:
df.emp_length.unique()

The `emp_length` we can ordinally encode since there is an incemental relation

In [ ]:
df["emp_length"] = df["emp_length"].map({"< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10})

In [ ]:
df.term.value_counts()

For the feature `term` we can strip the months and convert to integer

In [ ]:
df["term"] = df["term"].fillna("36 months")

In [ ]:
df["term"] = df["term"].str.strip().str.replace(" months", "").astype(int)

In [ ]:
df.purpose.value_counts().head()

In [ ]:
df['verification_status'].value_counts()

In [ ]:
df['initial_list_status'].value_counts()

In [ ]:
df['home_ownership'].value_counts()

We also know `sub_grade` is a more detailed version of `grade`, so we can drop `grade`. Since `sub_grade` is an ordered scale (A1 = best ... G5 = worst) we ordinal encode it rather than one-hot encode it; the remaining nominal categoricals are one-hot encoded.

In [ ]:
df.drop('grade', axis=1, inplace=True)

sub_grade_order = sorted(df['sub_grade'].unique()) 
df['sub_grade'] = df['sub_grade'].map({sg: i for i, sg in enumerate(sub_grade_order, start=1)})

dummies = ['verification_status', 'purpose', 'initial_list_status', 'home_ownership', 'addr_state']
df = pd.get_dummies(df, columns=dummies, drop_first=True)

In [ ]:
df.dtypes == "object"

In [ ]:
df['earliest_cr_line'].head()

In [ ]:
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y")

# Credit history length in years, relative to a fixed reference (dataset ~2018)
reference = pd.Timestamp("2018-01-01")
df["credit_history_yrs"] = (reference - df["earliest_cr_line"]).dt.days / 365.25

df.drop(columns="earliest_cr_line", inplace=True)

### Feature engineering
`fico_range_low` and `fico_range_high` are nearly identical, so we combine them into a single `fico_mid` feature.

In [ ]:
df["fico_mid"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
df.drop(columns=["fico_range_low", "fico_range_high"], inplace=True)



`annual_inc` and `revol_bal` are heavily right-skewed, so we log-transform them.

In [ ]:

df["annual_inc"] = np.log1p(df["annual_inc"])
df["revol_bal"] = np.log1p(df["revol_bal"])

We need to drop issue date since it is data leakage as the model would no know if the loan has been approved or not before making its prediction

In [ ]:
df.drop('issue_d', axis=1, inplace=True)


## Fill missing values

In [ ]:
df.isnull().sum()

We see that multiple features have rows with 33 null values. Are these the same rows?

In [ ]:
df[df['loan_amnt'].isnull() == True].isnull().sum()

Yess these rows contain a lot of null vallues lets drop them

In [ ]:
df = df.dropna(subset=['loan_amnt'])

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=["target"])

### This dataset does not have that much class imbalance (20%) compared to financial datasets. For our research we want class imbalance, so we will randomly undersample to get the minority class to 1%, 5%, 10% and 20%

Add a helper function to give us insights if undersampling introduced bias

In [ ]:
from scipy.stats import ks_2samp

def calc_bias(df, undersampled_df):
    results = {}
    for col in df.select_dtypes(include='number').columns:
        stat, p = ks_2samp(df[col], undersampled_df[col])
        results[col] = (stat, p)
    return results

In [ ]:
datasets = {}

minority_frac = [0.01, 0.05, 0.1, 0.2]

majority = df[df["target"] == 0]
minority = df[df["target"] == 1]

natural_frac = len(minority) / (len(minority) + len(majority))
print(f"Natural minority fraction: {natural_frac:.3f}")

for frac in minority_frac:
    if frac <= natural_frac:

        n_minority = int(round(frac / (1 - frac) * len(majority)))
        maj_part = majority
        min_part = minority.sample(n=n_minority, random_state=RANDOM_STATE)
    else:

        n_majority = int(round((1 - frac) / frac * len(minority)))
        maj_part = majority.sample(n=n_majority, random_state=RANDOM_STATE)
        min_part = minority

    dataset = (
        pd.concat([maj_part, min_part])
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
    X = dataset.drop(columns="target")
    y = dataset["target"]
    datasets[frac] = (X, y)
    print(f"Minority fraction: {frac}")
    print(y.value_counts())
    print(y.value_counts(normalize=True))
    print(f"Bias statistics (Kolmogorov-Smirnov test):")
    print(calc_bias(df, dataset))

# Section 3 resampling

For oversampling techinques we will use:
- Class weighting (cost-sensitive Learning)
- Random Under-Sampling
- SMOTE-ENN (Edited Nearest Neighbour)
- K-Means SMOTE

We compare this with the baseline (highly imbalanced) and using classweight


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import KMeansSMOTE
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

Define our samplers

In [ ]:
samplers = {
    "Baseline": None,   
    "RandomUnderSampler": RandomUnderSampler(sampling_strategy="majority", random_state=RANDOM_STATE),
    "SMOTEENN":           SMOTEENN(sampling_strategy="minority", random_state=RANDOM_STATE),
    "KMeansSMOTE": KMeansSMOTE(
        sampling_strategy="minority",
        random_state=RANDOM_STATE,
        cluster_balance_threshold=0.01,
        kmeans_estimator=MiniBatchKMeans(n_clusters=100, random_state=RANDOM_STATE, n_init=3),
    ),
    "ClassWeight":        None,
}

Define our Kfold

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Section 4 Model Implementation

We will implement the following models that have surfaced in the literature review:
- Random forest
- Logistic regression (since this is the most explainable model with the best performance)
- XGBoost (Black box tree based model)
- Neural Network

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from torch import nn

from sklearn.model_selection import cross_validate
from time import process_time
from sklearn.impute import SimpleImputer

import numpy as np
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, precision_recall_curve,
)
from imblearn.metrics import geometric_mean_score
from sklearn.preprocessing import FunctionTransformer

First define some helper functions

In [ ]:
results = []

def best_f1_threshold(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]

#Helper for NN models that expect float32 input
def to_float32(X):
    return np.asarray(X, dtype=np.float32)

We want to define our pipeline to pass into the 5kfold. We add the imputer for NaN values and scale the values

In [ ]:
def build_pipeline(estimator, sampler):
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
    if sampler is not None:
        steps.append(("sampler", sampler))
    if estimator.__class__.__name__ == "NeuralNetClassifier":
        steps.append(("to_float32", FunctionTransformer(to_float32)))
    steps.append(("clf", estimator))
    return ImbPipeline(steps)

We want to compute the SHAP scores for each training to see if resmapling methods alter feature importance

In [ ]:
import shap

shap_records = [] 

N_SHAP_MAX   = 200   # test samples for TreeExplainer / LinearExplainer
N_SHAP_NN    = 20    # test samples for NeuralNet (KernelExplainer is slow)
N_SHAP_NN_BG = 50    # background samples for KernelExplainer

In [ ]:
def preprocess_for_shap(pipe, X):
    # Remove resampler steps from the pipeline, as they are not used at inference time and may not support transform().
    X_t = X
    for _, step in pipe.steps[:-1]:
        if hasattr(step, "fit_resample"): 
            continue
        X_t = step.transform(X_t)
    return np.asarray(X_t)


def get_sv_class1(sv):
    # Normalise SHAP output to shape (n_samples, n_features) for the positive class.
    if isinstance(sv, list) and len(sv) == 2:
        return sv[1]
    if hasattr(sv, "values"):               # shap.Explanation object (SHAP >= 0.41)
        v = sv.values
        return v[:, :, 1] if v.ndim == 3 else v
    if isinstance(sv, np.ndarray) and sv.ndim == 3:
        return sv[:, :, 1]
    return sv


def compute_shap_for_fold(pipe, train_idx, test_idx, X, model_name, sampler_name, imbalance_ratio):
    rng = np.random.default_rng(RANDOM_STATE)
    clf  = pipe.named_steps["clf"]
    feat = list(X.columns)

    try:
        if model_name in ("XGBClassifier", "RandomForestClassifier"):
            X_te = preprocess_for_shap(pipe, X.iloc[test_idx])
            idx  = rng.choice(len(X_te), min(N_SHAP_MAX, len(X_te)), replace=False)
            sv   = get_sv_class1(shap.TreeExplainer(clf).shap_values(X_te[idx]))

        elif model_name == "LogisticRegression":
            X_tr = preprocess_for_shap(pipe, X.iloc[train_idx])
            X_te = preprocess_for_shap(pipe, X.iloc[test_idx])
            idx  = rng.choice(len(X_te), min(N_SHAP_MAX, len(X_te)), replace=False)
            sv   = get_sv_class1(shap.LinearExplainer(clf, X_tr).shap_values(X_te[idx]))

        elif model_name == "NeuralNetClassifier":
            X_te_raw = X.iloc[test_idx].astype(np.float32)
            X_tr_raw = X.iloc[train_idx].astype(np.float32)
            ei = rng.choice(len(X_te_raw), min(N_SHAP_NN,    len(X_te_raw)),  replace=False)
            bi = rng.choice(len(X_tr_raw), min(N_SHAP_NN_BG, len(X_tr_raw)), replace=False)

            def nn_predict(X_raw):
                df = pd.DataFrame(X_raw, columns=feat).astype(np.float32)
                return pipe.predict_proba(df)[:, 1]

            sv = shap.KernelExplainer(nn_predict, X_tr_raw.iloc[bi]).shap_values(
                X_te_raw.iloc[ei], nsamples=100, l1_reg="num_features(10)"
            )
        else:
            return

        shap_records.append({
            "Model":           model_name,
            "Sampler":         sampler_name,
            "Imbalance ratio": imbalance_ratio,
            "importances":     pd.Series(np.abs(sv).mean(axis=0), index=feat),
        })

    except Exception as e:
        print(f"  SHAP skipped for {model_name}+{sampler_name} (IR={imbalance_ratio}): {e}")

The scoring function

In [ ]:
def score_model(imbalance_ratio, cv_out, estimator, sampler_name, X, y, normalize='true'):
    y_arr = np.asarray(y)
    y_pred = np.empty_like(y_arr)
    fold_thresholds, train_f1s, test_f1s = [], [], []
    for est, train_idx, test_idx in zip(
        cv_out["estimator"], cv_out["indices"]["train"], cv_out["indices"]["test"]
    ):
        proba_train = est.predict_proba(X.iloc[train_idx])[:, 1]
        thr = best_f1_threshold(y_arr[train_idx], proba_train)
        proba_test = est.predict_proba(X.iloc[test_idx])[:, 1]
        y_pred[test_idx] = (proba_test >= thr).astype(y_arr.dtype)
        fold_thresholds.append(float(thr))
        train_f1s.append(f1_score(y_arr[train_idx], (proba_train >= thr).astype(y_arr.dtype), average="macro"))
        test_f1s.append(f1_score(y_arr[test_idx], y_pred[test_idx], average="macro"))

    cm = confusion_matrix(y_arr, y_pred, normalize=normalize)
    disp = ConfusionMatrixDisplay(cm, display_labels=np.unique(y_arr))
    disp.plot(cmap="Blues", values_format=".2f" if normalize else "d")
    plt.title(f"{estimator.__class__.__name__} + {sampler_name} + imbalance ratio {imbalance_ratio} ({skf.get_n_splits()}-fold OOF)")
    plt.show()

    train_f1 = float(np.mean(train_f1s))
    test_f1 = float(np.mean(test_f1s))

    results.append({
        "Model": estimator.__class__.__name__,
        "Sampler": sampler_name,
        "Imbalance ratio": imbalance_ratio,
        "fit time (s)": round(cv_out["fit_time"].mean(), 2),
        "Score time (s)": round(cv_out["score_time"].mean(), 2),
        "Threshold (mean)": round(float(np.mean(fold_thresholds)), 4),
        "F1 (mean)":        test_f1,
        "ROC-AUC (mean)":   cv_out["test_roc_auc"].mean(),
        "PR-AUC (mean)":    cv_out["test_average_precision"].mean(),
        "Recall (mean)":    recall_score(y_arr, y_pred, average="macro"),
        "Precision (mean)": precision_score(y_arr, y_pred, average="macro"),
        "G-Mean (mean)":    geometric_mean_score(y_arr, y_pred),
        "Train F1 (mean)":  train_f1,
        "F1 gap (train-test)": round(train_f1 - test_f1, 3),
        "TP": cm[1, 1] if normalize else int(cm[1, 1]),
        "FP": cm[0, 1] if normalize else int(cm[0, 1]),
        "TN": cm[0, 0] if normalize else int(cm[0, 0]),
        "FN": cm[1, 0] if normalize else int(cm[1, 0])
    })
    
    compute_shap_for_fold(
        cv_out["estimator"][-1],
        cv_out["indices"]["train"][-1],
        cv_out["indices"]["test"][-1],
        X,
        estimator.__class__.__name__,
        sampler_name,
        imbalance_ratio,
    )

Define our Kfold function

In [ ]:

def model_kfold(sampler_name, estimator, X, y, imbalance_ratio=None):
    print(f"Training {estimator.__class__.__name__} + {sampler_name} with imbalance ratio {imbalance_ratio}...")

    print("Performing cross-validation...")
    time_start = process_time()

    sampler = samplers[sampler_name]

    pipe = build_pipeline(estimator, sampler)

    cv_out = cross_validate(
        pipe, X, y, cv=skf,
        scoring=("roc_auc", "average_precision"),
        return_train_score=True,
        return_estimator=True,
        return_indices=True
    )

    print(f"Done in {round(process_time() - time_start, 2)} seconds.")

    score_model(imbalance_ratio, cv_out, estimator, sampler_name, X, y)

In [ ]:
import torch

Applying class weight is different per model so we have to define a function to compute this correctly

In [ ]:
def apply_class_weight(estimator, sampler_name, y):
    name = estimator.__class__.__name__
    y_arr = np.asarray(y)
    if sampler_name == "ClassWeight":
        if name in ("LogisticRegression", "RandomForestClassifier"):
            estimator.set_params(class_weight="balanced")
        elif name == "XGBClassifier":
            estimator.set_params(scale_pos_weight=(y_arr == 0).sum() / (y_arr == 1).sum())
        elif name == "NeuralNetClassifier":
            _, counts = np.unique(y_arr, return_counts=True)
            w = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32)
            estimator.set_params(criterion__weight=w)
    else:
        if name in ("LogisticRegression", "RandomForestClassifier"):
            estimator.set_params(class_weight=None)
        elif name == "XGBClassifier":
            estimator.set_params(scale_pos_weight=1.0)
        elif name == "NeuralNetClassifier":
            estimator.set_params(criterion__weight=None)
    return estimator

Now we implement our loops. First over all sampling methods, then over all imbalance ratios per model

In [ ]:
def loop_samplers(estimator, X, y, imbalance_ratio):
    for sampler_name, sampler in samplers.items(): 
        model_kfold(sampler_name, apply_class_weight(estimator, sampler_name, y), X, y, imbalance_ratio=imbalance_ratio)

In [ ]:
def loop_imbalance_ratios(estimator):
    for frac, dataset in datasets.items():
        print(f"=== Minority fraction: {frac} ===")
        X, y = dataset
        if estimator.__class__.__name__ == "NeuralNetClassifier":
            X = X.astype(np.float32)
            y = y.to_numpy().astype(np.int64)
            
        loop_samplers(estimator, X, y, imbalance_ratio=frac)

Before training on the various imbalance ratios we will first use gridsearch to find the optimal params for the models

In [ ]:
from sklearn.model_selection import GridSearchCV

config = {}

def find_best_hyperparameters(estimator, X, y, param_distributions):
    param_distributions["clf__random_state"] = [RANDOM_STATE]

    pipe = ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", estimator)
    ])

    search = GridSearchCV(
        pipe,
        param_distributions,
        scoring="average_precision",
        refit=True,
        n_jobs=-1
    )

    search.fit(X, y)

    print("Best hyperparameters:")
    print(search.best_params_)

    print(f"Score with best hyperparameters: {search.best_score_:.4f}")

    config[estimator.__class__.__name__] = {k.removeprefix("clf__"): v for k, v in search.best_params_.items()}

In [ ]:
weight = (datasets[0.2][1] == 0).sum() / (datasets[0.2][1] == 1).sum()

In [ ]:
send_telegram_message("Starting XGBoost training...")

In [ ]:

param_distributions_xgb = {
    "clf__n_estimators":     [400, 800],
    "clf__max_depth":        [4, 6],
    "clf__learning_rate":    [0.05, 0.1],
    "clf__min_child_weight": [1, 10],
    "clf__reg_lambda":       [1, 5],
    "clf__subsample":        [0.8],
    "clf__colsample_bytree": [0.8],
}


find_best_hyperparameters(XGBClassifier(), datasets[0.2][0], datasets[0.2][1], param_distributions_xgb)

In [ ]:
loop_imbalance_ratios(XGBClassifier(**config["XGBClassifier"]))

In [ ]:
send_telegram_message("Starting Logistic Regression training...")

In [ ]:
param_distributions_logistic= {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__penalty": ["l1", "l2"],
    "clf__solver": ["saga"],
    "clf__max_iter": [2000],
}

find_best_hyperparameters(LogisticRegression(), datasets[0.2][0], datasets[0.2][1], param_distributions_logistic)

In [ ]:
loop_imbalance_ratios(LogisticRegression(**config["LogisticRegression"]))

In [ ]:
send_telegram_message("Starting Random Forest training...")

In [ ]:
param_distributions_random_forest = {
    "clf__n_estimators":     [500],
    "clf__max_depth":        [None, 16, 24],
    "clf__min_samples_leaf": [1, 5, 20],     
    "clf__max_features":     ["sqrt", 0.3],
} 

find_best_hyperparameters(RandomForestClassifier(), datasets[0.2][0], datasets[0.2][1], param_distributions_random_forest)

In [ ]:
loop_imbalance_ratios(RandomForestClassifier(**config["RandomForestClassifier"]))

## Implement Neural Network

Define our neural network architecture

In [ ]:
class NeuralNetwork(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.layer_1 = nn.Linear(input_dim, 64)
        self.activation_1 = nn.ReLU()
        self.dropout_1 = nn.Dropout(0.2)
        self.layer_2 = nn.Linear(64, 64)
        self.activation_2 = nn.ReLU()
        self.dropout_2 = nn.Dropout(0.2)
        self.layer_3 = nn.Linear(64, 2)

    def forward(self, X):

        X = self.layer_1(X)
        X = self.activation_1(X)
        X = self.dropout_1(X)
        X = self.layer_2(X)
        X = self.activation_2(X)
        X = self.dropout_2(X)
        X = self.layer_3(X)

        return X


n_features = next(iter(datasets.values()))[0].shape[1]

Check if GPU is available

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
from skorch import NeuralNetClassifier
from skorch.callbacks import EarlyStopping, LRScheduler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import Adam

net_clf = NeuralNetClassifier(
    NeuralNetwork,
    module__input_dim=n_features,
    criterion=nn.CrossEntropyLoss,
    max_epochs=100,
    lr=0.001,
    device=device,
    optimizer=Adam,
    batch_size=512,
    iterator_train__shuffle=True,
    callbacks=[
    EarlyStopping(monitor='valid_loss', patience=10),
    LRScheduler(policy=ReduceLROnPlateau, monitor='valid_loss'),]
)

In [ ]:
send_telegram_message("Starting Neural Network training...")

In [ ]:
loop_imbalance_ratios(net_clf)

## Evaluation of models & resampling methods

In [ ]:
model_results = pd.DataFrame(results)
model_results.to_csv("model_results.csv", index=False)
model_results

In [ ]:
send_telegram_message("Evaluation started")

In [ ]:
ir_col   = "Imbalance ratio"
y_col    = "PR-AUC (mean)"

unique_samplers = model_results["Sampler"].unique()
models   = model_results["Model"].unique()

n = len(unique_samplers)
ncols = 3
nrows = -(-n // ncols) 

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows),
                         sharex=True, sharey=True)
axes = axes.flatten()

for ax, sampler in zip(axes, unique_samplers):
    sub = model_results[model_results["Sampler"] == sampler]
    for model in models:
        m = sub[sub["Model"] == model].sort_values(ir_col)
        if m.empty:
            continue
        ax.plot(m[ir_col], m[y_col], marker="o", label=model)

    ax.set_title(sampler)
    ax.set_xlabel("Imbalance Ratio")
    ax.set_ylabel(y_col)
    ax.grid(True, alpha=0.3)

for ax in axes[n:]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center",
           ncol=len(models), bbox_to_anchor=(0.5, 1.02))

fig.tight_layout()
plt.show()


## Section 5b: Global SHAP Analysis

Mean absolute SHAP values are collected inside each CV run using the last fold's already-fitted pipeline so no models are retrained. Results cover all combinations of model × sampler × imbalance ratio.


In [ ]:
from matplotlib.patches import Patch

In [ ]:

# Features the attacker is allowed to modify
mutable_features = [
    "loan_amnt", "annual_inc", "dti", "revol_bal",
    "revol_util", "open_acc", "total_acc",
]

TOP_N = 15

In [ ]:


shap_rows = []
for rec in shap_records:
    row = {"Model": rec["Model"], "Sampler": rec["Sampler"], "Imbalance ratio": rec["Imbalance ratio"]}
    row.update(rec["importances"].to_dict())
    shap_rows.append(row)

shap_wide = pd.DataFrame(shap_rows)
shap_wide.to_csv("shap_importances.csv", index=False)

feat_cols = [c for c in shap_wide.columns if c not in ("Model", "Sampler", "Imbalance ratio")]
shap_models_order = list(shap_wide["Model"].unique())

fig, axes = plt.subplots(1, len(shap_models_order), figsize=(6 * len(shap_models_order), 6))
axes = np.atleast_1d(axes)

for ax, model_name in zip(axes, shap_models_order):
    imp = shap_wide[shap_wide["Model"] == model_name][feat_cols].mean().nlargest(TOP_N)
    colors = ["tab:orange" if f in mutable_features else "tab:blue" for f in imp.index]
    imp[::-1].plot(kind="barh", ax=ax, color=colors[::-1])
    ax.set_title(model_name)
    ax.set_xlabel("Mean |SHAP value|")
fig.legend(
    handles=[
        Patch(facecolor="tab:orange", label="Adversarially mutable"),
        Patch(facecolor="tab:blue",   label="Not mutable"),
    ],
    loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04),
)
fig.suptitle(
    f"Top {TOP_N} Global SHAP Features per Model  (averaged across samplers & imbalance ratios)",
    y=1.06,
)
plt.tight_layout()
plt.show()

for model_name in shap_models_order:
    sub  = shap_wide[shap_wide["Model"] == model_name]
    top5 = sub[feat_cols].mean().nlargest(5).index.tolist()

    fig, ax = plt.subplots(figsize=(8, 4))
    for feat in top5:
        grp = sub.groupby("Imbalance ratio")[feat].mean()
        ax.plot(grp.index, grp.values, marker="o", label=feat)
    ax.set_title(f"{model_name} — Top-5 SHAP features vs Imbalance Ratio")
    ax.set_xlabel("Imbalance Ratio")
    ax.set_ylabel("Mean |SHAP value|")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Section 5c: Fairness / Disparate Impact Ratio (DIR)

This section evaluates whether the trained model gives the favourable prediction at different rates across groups.

- `emp_length`: shorter employment length as unprivileged, longer employment length as privileged.
- `annual_inc`: lower income as unprivileged, higher income as privileged.

Thresholds are fitted on the training split and then applied to the test split to avoid test-set leakage.

In [ ]:
# DIR configuration and helper functions
from sklearn.base import clone
from sklearn.metrics import accuracy_score

# target = 1 means default, so the favourable prediction is non-default.
FAVOURABLE_LABEL = 0

# For numerical features, use either:
# - method="threshold" with a fixed threshold, or
# - method="quantile" with a training-set quantile.
DIR_GROUP_SPECS = {
    "emp_length": {
        "method": "threshold",
        "threshold": 5,
        "unprivileged_if": "lower",   # lower than threshold = unprivileged
        "unprivileged_name": "emp_length < 5 years",
        "privileged_name": "emp_length >= 5 years",
    },
    "annual_inc": {
        "method": "quantile",
        "quantile": 0.50,             # median split based on the training set
        "unprivileged_if": "lower",
        "unprivileged_name": "lower annual income",
        "privileged_name": "higher annual income",
    },
}


def _get_group_threshold(feature, spec, X_reference):
    if spec["method"] == "threshold":
        return spec["threshold"]
    if spec["method"] == "quantile":
        return X_reference[feature].quantile(spec.get("quantile", 0.50))


def make_binary_groups(feature, values, spec, threshold):
    """Return privileged/unprivileged group labels for one feature."""
    if spec.get("unprivileged_if", "lower") == "lower":
        unprivileged_mask = values < threshold
    elif spec.get("unprivileged_if") == "higher":
        unprivileged_mask = values >= threshold

    return pd.Series(
        np.where(unprivileged_mask, spec["unprivileged_name"], spec["privileged_name"]),
        index=values.index,
        name=feature,
    )


def evaluate_dir_for_groups(
    y_true,
    y_pred,
    X_eval,
    X_reference,
    group_specs=DIR_GROUP_SPECS,
    favourable_label=FAVOURABLE_LABEL,
):
    y_true = pd.Series(np.asarray(y_true), index=X_eval.index, name="y_true")
    y_pred = pd.Series(np.asarray(y_pred), index=X_eval.index, name="y_pred")
    favourable_pred = (y_pred == favourable_label).astype(int)

    rows = []
    for feature, spec in group_specs.items():
        if feature not in X_eval.columns:
            print(f"Skipping {feature}: not found in X_eval")
            continue

        threshold = _get_group_threshold(feature, spec, X_reference)
        groups = make_binary_groups(feature, X_eval[feature], spec, threshold)

        tmp = pd.DataFrame({
            "group": groups,
            "y_true": y_true,
            "y_pred": y_pred,
            "favourable_pred": favourable_pred,
        })

        group_table = tmp.groupby("group").apply(
            lambda g: pd.Series({
                "n_samples": len(g),
                "favourable_prediction_rate": g["favourable_pred"].mean(),
                "default_prediction_rate": (g["y_pred"] == 1).mean(),
                "accuracy": accuracy_score(g["y_true"], g["y_pred"]),
            })
        )

        unpriv = spec["unprivileged_name"]
        priv = spec["privileged_name"]

        # Avoid divide-by-zero when the privileged group has no favourable predictions.
        priv_rate = group_table.loc[priv, "favourable_prediction_rate"] if priv in group_table.index else np.nan
        unpriv_rate = group_table.loc[unpriv, "favourable_prediction_rate"] if unpriv in group_table.index else np.nan
        dir_score = unpriv_rate / priv_rate if pd.notna(priv_rate) and priv_rate > 0 else np.nan

        for group_name, vals in group_table.iterrows():
            rows.append({
                "Feature": feature,
                "Group": group_name,
                "Role": "unprivileged" if group_name == unpriv else "privileged",
                "Threshold used": threshold,
                "DIR": dir_score,
                **vals.to_dict(),
            })

    return pd.DataFrame(rows)


In [ ]:
# Run DIR evaluation for each imbalance ratio, sampler, and selected model.
# To save time, start with XGBoost only. Add other models here when needed.

xgb_params = config.get("XGBClassifier", {}).copy() if "config" in globals() else {}
xgb_params.setdefault("random_state", RANDOM_STATE)
xgb_params.setdefault("eval_metric", "logloss")

FAIRNESS_MODELS = [
    XGBClassifier(**xgb_params),
    # LogisticRegression(**config["LogisticRegression"]),
    # RandomForestClassifier(**config["RandomForestClassifier"]),
    # net_clf,
]

fairness_rows = []

for frac, (X_full, y_full) in datasets.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_full,
        y_full,
        test_size=0.2,
        stratify=y_full,
        random_state=RANDOM_STATE,
    )

    for model in FAIRNESS_MODELS:
        model_name = model.__class__.__name__
        is_nn = model_name == "NeuralNetClassifier"
        y_fit = y_tr.to_numpy().astype(np.int64) if is_nn else y_tr

        for sampler_name, sampler in samplers.items():
            print(f"DIR evaluation: imbalance={frac}, model={model_name}, sampler={sampler_name}")

            try:
                est = apply_class_weight(clone(model), sampler_name, y_fit)
                pipe = build_pipeline(est, sampler)
                pipe.fit(X_tr, y_fit)

                y_pred = pipe.predict(X_te)

                dir_df = evaluate_dir_for_groups(
                    y_true=y_te,
                    y_pred=y_pred,
                    X_eval=X_te,
                    X_reference=X_tr,
                    group_specs=DIR_GROUP_SPECS,
                    favourable_label=FAVOURABLE_LABEL,
                )

                dir_df.insert(0, "Imbalance ratio", frac)
                dir_df.insert(1, "Model", model_name)
                dir_df.insert(2, "Sampler", sampler_name)
                fairness_rows.append(dir_df)

            except Exception as e:
                fairness_rows.append(pd.DataFrame([{
                    "Imbalance ratio": frac,
                    "Model": model_name,
                    "Sampler": sampler_name,
                    "Feature": np.nan,
                    "Group": np.nan,
                    "Role": np.nan,
                    "Threshold used": np.nan,
                    "DIR": np.nan,
                    "n_samples": np.nan,
                    "favourable_prediction_rate": np.nan,
                    "default_prediction_rate": np.nan,
                    "accuracy": np.nan,
                    "Error": str(e),
                }]))
                print(f"FAILED: imbalance={frac}, model={model_name}, sampler={sampler_name}: {e}")

fairness_results_df = pd.concat(fairness_rows, ignore_index=True)
fairness_results_df.to_csv("dir_fairness_results.csv", index=False)
fairness_results_df.round(3)


In [ ]:
# Compact DIR table: one row per model/sampler/imbalance-ratio/feature
# Includes privileged and unprivileged favourable prediction rates.

dir_summary = (
    fairness_results_df
    .dropna(subset=["Feature"])
    .pivot_table(
        index=["Imbalance ratio", "Model", "Sampler", "Feature", "DIR", "Threshold used"],
        columns="Role",
        values=["favourable_prediction_rate", "default_prediction_rate", "n_samples", "accuracy"],
        aggfunc="first"
    )
)

dir_summary.columns = [
    "_".join(col).strip()
    for col in dir_summary.columns.to_flat_index()
]

dir_summary = dir_summary.reset_index()

dir_summary["Potential disparate impact (<0.8)"] = dir_summary["DIR"] < 0.8

dir_summary = dir_summary.sort_values(
    ["Feature", "Sampler", "Imbalance ratio"]
)

dir_summary.round(3)

# Section 6: Adversarial Attacks across all models, samplers and imbalance ratios

We run the constrained random-search counterfactual attack against **every combination** of
model (XGBoost, Logistic Regression, Random Forest, Neural Network), resampling method
(Baseline, RandomUnderSampler, SMOTEENN, KMeansSMOTE, ClassWeight) and imbalance ratio.

For each combination we refit the pipeline on a stratified train split, attack the
correctly-predicted defaults from the held-out test split, and report the attack metrics
(success rate, mean changed features, mean probability reduction).

In [ ]:
N_ATTACK_CASES = 50      
ATTACK_MAX_ITER = 100    
N_CHANGES = 1        

In [ ]:
def constrained_random_attack(
    model,
    x_original,
    X_reference,
    mutable_features,
    desired_class=0,
    max_iter=500,
    n_changes=1,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    x_original = x_original.copy()
    original_prob = model.predict_proba(x_original)[:, 1][0]

    best_x = x_original.copy()
    best_prob = original_prob
    best_pred = model.predict(x_original)[0]
    best_changes = {}

    for _ in range(max_iter):
        x_candidate = x_original.copy()

        chosen_features = rng.choice(
            mutable_features,
            size=min(n_changes, len(mutable_features)),
            replace=False
        )

        changes = {}

        for feature in chosen_features:
            old_value = x_candidate[feature].iloc[0]

            sampled_value = X_reference[feature].sample(
                n=1,
                random_state=int(rng.integers(0, 1_000_000))
            ).values[0]

            x_candidate.loc[:, feature] = sampled_value

            if old_value != sampled_value:
                changes[feature] = {
                    "old": old_value,
                    "new": sampled_value
                }

        pred = model.predict(x_candidate)[0]
        prob = model.predict_proba(x_candidate)[:, 1][0]

        if prob < best_prob:
            best_prob = prob
            best_x = x_candidate.copy()
            best_pred = pred
            best_changes = changes.copy()

        if pred == desired_class:
            changed_features = (x_original.iloc[0] != x_candidate.iloc[0]).sum()

            return {
                "success": 1,
                "cf_instance": x_candidate,
                "cf_pred": pred,
                "cf_default_prob": prob,
                "changed_features": changed_features,
                "prob_reduction": original_prob - prob,
                "changes": changes
            }

    changed_features = (x_original.iloc[0] != best_x.iloc[0]).sum()

    return {
        "success": 0,
        "cf_instance": best_x,
        "cf_pred": best_pred,
        "cf_default_prob": best_prob,
        "changed_features": changed_features,
        "prob_reduction": original_prob - best_prob,
        "changes": best_changes
    }

In [ ]:
send_telegram_message("Adversarial attacks started.")

In [ ]:

models = [
    XGBClassifier(**config["XGBClassifier"]),
    LogisticRegression(**config["LogisticRegression"]),
    RandomForestClassifier(**config["RandomForestClassifier"]),
    net_clf,
]

adversarial_results = []

for frac, (X_full, y_full) in datasets.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_full, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_STATE
    )
    feats = [f for f in mutable_features if f in X_tr.columns]
    input_dim = X_tr.shape[1]

    for model in models:
        is_nn = model.__class__.__name__ == "NeuralNetClassifier"
        y_fit = y_tr.to_numpy().astype(np.int64) if is_nn else y_tr

        for sampler_name, sampler in samplers.items():
            est = apply_class_weight(model, sampler_name, y_fit)
            pipe = build_pipeline(
                est, sampler
            )

            row = {
                "Imbalance ratio": frac,
                "Model": model.__class__.__name__,
                "Sampler": sampler_name,
            }

            try:
                pipe.fit(X_tr, y_fit)

                # attack candidates: true defaults that are correctly predicted as default
                y_pred_te = pipe.predict(X_te)
                mask = (y_te.to_numpy() == 1) & (np.asarray(y_pred_te) == 1)
                X_cand = X_te.loc[mask]
                n_cases = min(N_ATTACK_CASES, len(X_cand))

                if n_cases == 0:
                    row.update({
                        "N attacked": 0,
                        "Attack success rate": np.nan,
                        "Mean changed features": np.nan,
                        "Mean prob reduction (success)": np.nan,
                        "Mean prob reduction (all)": np.nan,
                    })
                    print(f"frac={frac} {model.__class__.__name__}+{sampler_name}: no correct-default candidates")
                    adversarial_results.append(row)
                    continue

                X_cand = X_cand.sample(n=n_cases, random_state=RANDOM_STATE)

                case_results = []
                for idx in X_cand.index:
                    x0 = X_te.loc[[idx]]
                    case_results.append(
                        constrained_random_attack(
                            model=pipe,
                            x_original=x0,
                            X_reference=X_tr,
                            mutable_features=feats,
                            desired_class=0,
                            max_iter=ATTACK_MAX_ITER,
                            n_changes=N_CHANGES,
                            random_state=RANDOM_STATE + int(idx),
                        )
                    )

                rdf = pd.DataFrame(case_results)
                succ = rdf["success"] == 1
                row.update({
                    "N attacked": n_cases,
                    "Attack success rate": rdf["success"].mean(),
                    "Mean changed features": rdf.loc[succ, "changed_features"].mean(),
                    "Mean prob reduction (success)": rdf.loc[succ, "prob_reduction"].mean(),
                    "Mean prob reduction (all)": rdf["prob_reduction"].mean(),
                })
                print(
                    f"frac={frac} {model.__class__.__name__}+{sampler_name}: "
                    f"success={row['Attack success rate']:.2f} ({n_cases} cases)"
                )

            except Exception as e:
                row.update({
                    "N attacked": np.nan,
                    "Attack success rate": np.nan,
                    "Mean changed features": np.nan,
                    "Mean prob reduction (success)": np.nan,
                    "Mean prob reduction (all)": np.nan,
                    "Error": str(e),
                })
                print(f"frac={frac} {model.__class__.__name__}+{sampler_name}: FAILED ({e})")

            adversarial_results.append(row)

In [ ]:
adversarial_attack_results_df = pd.DataFrame(adversarial_results)
adversarial_attack_results_df.to_csv("adversarial_attack_results.csv", index=False)
adversarial_attack_results_df

In [ ]:

adf = adversarial_attack_results_df
ir_col, y_col = "Imbalance ratio", "Attack success rate"

samplers_list = adf["Sampler"].unique()
models_list = adf["Model"].unique()

n = len(samplers_list)
ncols = 3
nrows = -(-n // ncols)

fig, axes = plt.subplots(
    nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), sharex=True, sharey=True
)
axes = np.atleast_1d(axes).flatten()

for ax, sampler in zip(axes, samplers_list):
    sub = adf[adf["Sampler"] == sampler]
    for model in models_list:
        m = sub[sub["Model"] == model].sort_values(ir_col)
        if m.empty:
            continue
        ax.plot(m[ir_col], m[y_col], marker="o", label=model)
    ax.set_title(sampler)
    ax.set_xlabel("Imbalance ratio")
    ax.set_ylabel(y_col)
    ax.grid(True, alpha=0.3)

for ax in axes[n:]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(models_list), bbox_to_anchor=(0.5, 1.02))

fig.tight_layout()
plt.show()

Add Telegram notification in case notebook is finished

In [ ]:
send_telegram_message("Model training and evaluation completed! Results saved to 'model_results.csv' and 'adversarial_attack_results.csv'.")